# CG-ROVER-CRF абляциясы: дауыс беру салмақтары және контекстсіз селектор
**Алдымен** `MAPP_ASR_KSC2_chunk_VAD`, `MAPP_ASR_MMS_VAD` т.б. арқылы жасалған 5 ASR шығысы Drive-та тұруы керек (бұрынғы `MAPP2_ROVER_CRF.ipynb`-тегідей). Ноутбук ASR-ды қайта жүргізбейді.

1-тәжірибе: дауыс беру салмақтарының бес схемасы + вложенная CV ішінде таңдау (таңдау тек оқыту файлдарындағы дауыс беру қателігімен жасалады).
2-тәжірибе: CRF-ті сол белгілермен әр слотты дербес таңдайтын логистикалық регрессиямен және градиентті бустингпен салыстыру (guard τ бірдей).

Күтілетін негізгі нәтиже: дауыс беру 18,28%, CRF 17,93% (мақаладағы).

In [1]:
!pip -q install sklearn-crfsuite jiwer
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.2 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
%%writefile mapp2_core.py
import math, re, random
from collections import Counter
import jiwer, sklearn_crfsuite

EPS = "<eps>"; MAXC = 4

def norm_text(t):
    t = str(t).lower(); t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()

def _ngram_dedup(w, k):
    o = []; i = 0
    while i < len(w):
        if len(o) >= k and w[i:i + k] == o[-k:]: i += k; continue
        o.append(w[i]); i += 1
    return o

def dedup_full(t):
    w = norm_text(t).split(); o = []; run = 0
    for x in w:
        if o and o[-1] == x: continue
        o.append(x)
    for k in (3, 2): o = _ngram_dedup(o, k)
    return " ".join(o)

# ---------- ROVER ----------
def add_to_network(net, hyp, nsys):
    if not net: return [[w] for w in hyp]
    n, m = len(net), len(hyp); INF = 10**9
    D = [[INF]*(m+1) for _ in range(n+1)]; B = [[None]*(m+1) for _ in range(n+1)]; D[0][0] = 0
    for i in range(n+1):
        for j in range(m+1):
            d = D[i][j]
            if d >= INF: continue
            if i < n and j < m:
                c = d + (0 if hyp[j] in net[i] else 1)
                if c < D[i+1][j+1]: D[i+1][j+1], B[i+1][j+1] = c, "M"
            if i < n:
                c = d + (0 if all(w == EPS for w in net[i]) else 1)
                if c < D[i+1][j]: D[i+1][j], B[i+1][j] = c, "D"
            if j < m:
                c = d + 1
                if c < D[i][j+1]: D[i][j+1], B[i][j+1] = c, "I"
    out, i, j = [], n, m
    while i > 0 or j > 0:
        op = B[i][j]
        if op == "M": out.append(net[i-1] + [hyp[j-1]]); i -= 1; j -= 1
        elif op == "D": out.append(net[i-1] + [EPS]); i -= 1
        else: out.append([EPS]*nsys + [hyp[j-1]]); j -= 1
    return out[::-1]

def build_network(hyps):
    net = []
    for s, h in enumerate(hyps): net = add_to_network(net, h.split(), s)
    return net

def slot_candidates(slot, sw):
    score, first = Counter(), {}
    for s, w in enumerate(slot): score[w] += sw[s]; first.setdefault(w, s)
    cands = sorted(score, key=lambda w: (-score[w], first[w]))[:MAXC]
    tot = sum(sw)
    return [{"w": w, "votes": score[w]/tot, "from_sys0": int(slot[0] == w)} for w in cands]

def align_ref(cps, ref):
    n, m = len(cps), len(ref); INF = 10**9
    words = [[c["w"] for c in cs] for cs in cps]
    D = [[INF]*(m+1) for _ in range(n+1)]; B = [[None]*(m+1) for _ in range(n+1)]; D[0][0] = 0
    for i in range(n+1):
        for j in range(m+1):
            d = D[i][j]
            if d >= INF: continue
            if i < n and j < m:
                c = d + (0 if ref[j] in words[i] else 1)
                if c < D[i+1][j+1]: D[i+1][j+1], B[i+1][j+1] = c, "M"
            if i < n:
                c = d + (0 if EPS in words[i] else 1)
                if c < D[i+1][j]: D[i+1][j], B[i+1][j] = c, "D"
            if j < m:
                c = d + 1
                if c < D[i][j+1]: D[i][j+1], B[i][j+1] = c, "I"
    gold, i, j = [None]*n, n, m
    while i > 0 or j > 0:
        op = B[i][j]
        if op == "M": gold[i-1] = ref[j-1]; i -= 1; j -= 1
        elif op == "D": gold[i-1] = EPS; i -= 1
        else: j -= 1
    return [f"C{ws.index(g)}" if g in ws else "C0" for ws, g in zip(words, gold)]

# ---------- features ----------
class MLMScorer:
    def __init__(self, name="xlm-roberta-base", window=15):
        import torch
        from transformers import AutoTokenizer, AutoModelForMaskedLM
        self.torch = torch; self.tok = AutoTokenizer.from_pretrained(name)
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.m = AutoModelForMaskedLM.from_pretrained(name).to(self.dev).eval()
        self.window = window; self.cache = {}
    def score(self, left, cand, right):
        key = (tuple(left[-self.window:]), cand, tuple(right[:self.window]))
        if key in self.cache: return self.cache[key]
        t = self.tok
        L = t(" ".join(left[-self.window:]), add_special_tokens=False)["input_ids"] if left else []
        R = t(" ".join(right[:self.window]), add_special_tokens=False)["input_ids"] if right else []
        C = t(" " + cand, add_special_tokens=False)["input_ids"]
        ids = [t.cls_token_id] + L + [t.mask_token_id]*len(C) + R + [t.sep_token_id]
        with self.torch.inference_mode():
            lg = self.m(self.torch.tensor([ids], device=self.dev)).logits[0]
        lp = self.torch.log_softmax(lg[1+len(L):1+len(L)+len(C)], -1)
        s = float(sum(lp[k, c] for k, c in enumerate(C)) / max(1, len(C)))
        self.cache[key] = s; return s

PRON = {"мен":"1sg","сен":"2sg","біз":"1pl","сіз":"2pol","сендер":"2pl","сіздер":"2pol"}
SUFFIX = [("2pol",("дыңыз","діңіз","тыңыз","тіңіз","сыздар","сіздер","сыз","сіз")),
          ("1pl",("мыз","міз","пыз","піз","быз","біз","дық","дік","тық","тік","сақ","сек")),
          ("1sg",("мын","мін","пын","пін","бын","бін","дым","дім","тым","тім","сам","сем")),
          ("2sg",("сың","сің","дың","дің","тың","тің","саң","сең"))]
def person_of(w):
    if w in PRON: return None
    for p, s in SUFFIX:
        if w.endswith(s): return p
def default_morph(word, left, right):
    p = person_of(word); subj = next((PRON[w] for w in reversed(left[-6:]) if w in PRON), None)
    return {"agree": 1.0 if p == subj else -1.0} if p and subj else {}

def slot_features(cands, left, right, mlm, morph, lex):
    f = {"bias": 1.0, "n_cand": float(len(cands))}
    if len(cands) == 1: return f
    bs = {}
    if mlm:
        for k, c in enumerate(cands):
            if c["w"] != EPS: bs[k] = mlm.score(left, c["w"], right)
    top = max(bs.values()) if bs else 0.0
    for k, c in enumerate(cands):
        p = f"c{k}_"; f[p+"votes"] = c["votes"]; f[p+"sys0"] = float(c["from_sys0"])
        if c["w"] == EPS: f[p+"eps"] = 1.0; continue
        f[p+"len"] = min(len(c["w"]), 15)/15
        if k in bs:
            f[p+"mlm"] = bs[k]; f[p+"mlm_rel"] = bs[k]-top; f[p+"mlm_best"] = float(bs[k] == top)
        if lex is not None:
            f[p+"known"] = float(c["w"] in lex); f[p+"logfreq"] = math.log1p(lex.get(c["w"], 0))/15
        for k2, v in morph(c["w"], left, right).items(): f[p+"m_"+k2] = float(v)
    return f

def featurize(net, sw, mlm, morph, lex):
    cps = [slot_candidates(s, sw) for s in net]
    maj = [c[0]["w"] for c in cps]; feats = []
    for i, cands in enumerate(cps):
        left = [w for w in maj[max(0, i-20):i] if w != EPS]
        right = [w for w in maj[i+1:i+21] if w != EPS]
        feats.append(slot_features(cands, left, right, mlm, morph, lex))
    return cps, feats

def decode(cps, marg, tau):
    out = []
    for cands, m in zip(cps, marg):
        k = 0
        if len(cands) > 1:
            best = max((l for l in m if int(l[1:]) < len(cands)), key=m.get)
            if best != "C0" and m[best] >= tau: k = int(best[1:])
        if cands[k]["w"] != EPS: out.append(cands[k]["w"])
    return " ".join(out)

def vote_text(cps): return " ".join(c[0]["w"] for c in cps if c[0]["w"] != EPS)

def errs(ref, hyp):
    o = jiwer.process_words(ref, hyp if hyp.strip() else "<empty>")
    return o.substitutions + o.deletions + o.insertions

def train_crf(data, files):
    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=200, all_possible_transitions=True)
    crf.fit([data[f][1] for f in files], [data[f][2] for f in files]); return crf

def split_folds(files, k, seed=0):
    o = files[:]; random.Random(seed).shuffle(o); return [o[i::k] for i in range(k)]

def choose_tau(data, refs, train, taus, inner=4, brk=2.0, min_gain=1):
    """τ по внутренней CV на train: максимизируем fix - brk*break (в слов-ошибках файла)."""
    folds = split_folds(train, inner, seed=1)
    margs = {}
    for te in folds:
        tr = [f for f in train if f not in set(te)]
        crf = train_crf(data, tr)
        for f in te: margs[f] = crf.predict_marginals_single(data[f][1])
    best, bt = 0.0, 1e9
    for t in taus:
        gain = 0.0
        for f in train:
            cps = data[f][0]; r = refs[f]
            base = errs(r, vote_text(cps)); new = errs(r, decode(cps, margs[f], t))
            d = base - new; gain += d if d > 0 else brk*d
        if gain > best + 1e-9 and gain >= min_gain: best, bt = gain, t
    return bt

def run_cv(files, refs, hyps, sw, mlm=None, morph=default_morph, lex=None,
           folds=5, taus=(0.4,0.5,0.6,0.7,0.8,0.9,0.95), log=print):
    data = {}
    for n, f in enumerate(files, 1):
        net = build_network(hyps[f]); cps, X = featurize(net, sw, mlm, morph, lex)
        data[f] = (cps, X, align_ref(cps, refs[f].split()))
        if n % 20 == 0: log(f"  features {n}/{len(files)}")
    out = {"raw": {}, "vote": {}, "crf_auto": {}, **{f"crf_{t}": {} for t in taus}}
    tau_used = []
    for k, te in enumerate(split_folds(files, folds)):
        tr = [f for f in files if f not in set(te)]
        crf = train_crf(data, tr); tau = choose_tau(data, refs, tr, taus); tau_used.append(tau)
        for f in te:
            cps, X, _ = data[f]; m = crf.predict_marginals_single(X)
            out["raw"][f] = hyps[f][0]; out["vote"][f] = vote_text(cps)
            out["crf_auto"][f] = decode(cps, m, tau)
            for t in taus: out[f"crf_{t}"][f] = decode(cps, m, t)
        log(f"  fold {k+1}/{folds} τ={tau}")
    out["oracle"] = {f: " ".join(c[int(l[1:])]["w"] for c, l in zip(data[f][0], data[f][2])
                                 if c[int(l[1:])]["w"] != EPS) for f in files}
    return out, tau_used, data

def summarize(files, refs, out):
    N = sum(len(refs[f].split()) for f in files)
    base = {f: errs(refs[f], out["raw"][f]) for f in files}
    rows = []
    for name, h in out.items():
        e = {f: errs(refs[f], h[f]) for f in files}
        cer = jiwer.cer([refs[f] for f in files], [h[f] or "x" for f in files])
        b = sum(e[f] < base[f] for f in files); w = sum(e[f] > base[f] for f in files)
        rows.append(dict(system=name, WER=sum(e.values())/N, CER=cer, better=b, worse=w, same=len(files)-b-w))
    return rows


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- CSE морфология (түбір + жалғау талдаушы) ----------
import re as _re_morph
import numpy as np
from collections import defaultdict as _dd, Counter as _Counter

MOTION = {"бар","кел","кет","шық","кір","қайт","жет","ұш","жүр","жүгір","жүз","көш","аттан","қон","түс","өт","апар","әкел","әкет","орал","жылжы","көтеріл","жақында","ауыс"}
TRANSITIVE = {"көр","ал","бер","жаса","оқы","жаз","айт","же","іш","тап","сал","қой","аш","жап","тыңда","түсін","біл","ұста","сат","кес","жу","таңда","шеш","көтер","тексер","дайында","аяқта","баста","бітір","орында"}
_HOMO = {"i":"і","a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","ј":"і","h":"һ"}
_VPAIR = {"а":"е","е":"а","о":"ө","ө":"о","ұ":"ү","ү":"ұ","ы":"і","і":"ы"}
def _mclean(s): return "".join(_HOMO.get(c, c) for c in str(s).strip().lower())
def _mtwin(e): return "".join(_VPAIR.get(c, c) for c in e)
CASES = ["nom","gen","dat","acc","loc","abl","ins"]
CASE_TAG = {"gen":1,"dat":2,"acc":3,"loc":4,"abl":5,"ınst":6,"ınstr":6,"instr":6,"inst":6}
FINITE = {"pst","per","tpat","ıft","tft","aft","apt","tpt","ppt","mdes","mimp","mcond"}

class Tagger:
    def __init__(self, stems, ana, votes):
        self.S, self.ana, self.votes = stems, ana, votes
        self.maxe = max((len(e) for e in ana), default=0); self._w = {}
    @classmethod
    def build(cls, stems_xlsx, endings_xlsx, uni_counter):
        import pandas as _pd
        st = _pd.read_excel(stems_xlsx, header=None)
        S = {_mclean(v) for v in st.iloc[:, 0].dropna()}
        S = {w for w in S if w and all(c.isalpha() or c == "-" for c in w)}
        en = _pd.read_excel(endings_xlsx, header=None, skiprows=1)
        ana = _dd(set)
        for e, d in zip(en.iloc[:, 0], en.iloc[:, 1]):
            if not isinstance(e, str) or not isinstance(d, str): continue
            tags = frozenset(_re_morph.findall(r"<([^>]+)>", d))
            if not tags: continue
            key = _mclean(e)
            for k in (key, _mtwin(key)):
                if k: ana[k].add(tags)
        ana = {k: list(v) for k, v in ana.items()}
        votes = cls._harvest(S, ana, uni_counter)
        print("CSE: түбір %d, жалғау-үлгі %d" % (len(S), len(ana)))
        return cls(S, ana, votes)
    @staticmethod
    def _harvest(S, ana, uni_counter):
        vb = {e for e, a in ana.items() if all("VB" in t for t in a)}
        nb = {e for e, a in ana.items() if all("NB" in t for t in a)}
        maxe = max((len(e) for e in ana), default=0)
        votes = _dd(lambda: [0, 0])
        for w, n in uni_counter.items():
            if len(w) < 4 or not w.isalpha(): continue
            for k in range(len(w) - 1, 1, -1):
                if len(w) - k > maxe: break
                stem, end = w[:k], w[k:]
                if stem not in S: continue
                if end in vb: votes[stem][0] += n
                elif end in nb: votes[stem][1] += n
        return {k: tuple(v) for k, v in votes.items()}
    def stem_pos(self, s):
        v, n = self.votes.get(s, (0, 0)); t = v + n
        if t < 5: return 2
        r = v / t
        return 1 if r > 0.6 else (0 if r < 0.4 else 2)
    def word(self, w):
        t = self._w.get(w)
        if t is None: t = self._analyze(w); self._w[w] = t
        return t
    def _analyze(self, w):
        if not w or not w.isalpha(): return (2, 0, False, False, False, False, False)
        if w in self.S:
            p = self.stem_pos(w); return (p, 0, False, p == 1, w in MOTION, w in TRANSITIVE, True)
        best, best_sc = None, -1e9
        for k in range(len(w) - 1, 1, -1):
            if len(w) - k > self.maxe: break
            stem, end = w[:k], w[k:]
            if stem not in self.S: continue
            for tags in self.ana.get(end, ()):
                sp = self.stem_pos(stem); sc = 0.1 * len(end)
                if "VB" in tags: sc += 2.0 if sp == 1 else (-1.0 if sp == 0 else 0.0)
                if "NB" in tags: sc += 2.0 if sp == 0 else (-1.0 if sp == 1 else 0.0)
                if sc > best_sc: best, best_sc = (stem, tags, sp), sc
        if best is None: return (2, 0, False, False, False, False, False)
        stem, tags, sp = best
        pos = 1 if ("VB" in tags and "NB" not in tags) else (0 if ("NB" in tags and "VB" not in tags) else sp)
        case = 0
        for t in tags:
            if t in CASE_TAG: case = CASE_TAG[t]; break
        poss = "pos" in tags
        fin = pos == 1 and bool(tags & FINITE) and "pp" not in tags
        return (pos, case, poss, fin, stem in MOTION, stem in TRANSITIVE, True)

CSE_TR_NAMES = (["pos_%s_%s" % (a, b) for a in range(3) for b in range(3)]
                + ["%s_poss" % c for c in CASES] + ["%s_verb" % c for c in CASES]
                + ["dat_motion", "acc_trans", "nom_fin", "same_case", "both_bad", "gen_nonposs"])

def _pair_feats(a, b):
    """(алдыңғы сөз, келесі сөз) талдауынан GEN+POSS, DAT+MOTION, ACC+TRANS т.б. белгілер."""
    pa, ca, _, _, _, _, oka = a; pb, cb, possb, finb, motb, trb, okb = b
    f = {}
    f["pos_%d_%d" % (pa, pb)] = 1.0
    if possb: f["%s_poss" % CASES[ca]] = 1.0
    if pb == 1: f["%s_verb" % CASES[ca]] = 1.0
    if ca == 2 and motb: f["dat_motion"] = 1.0
    if ca == 3 and trb: f["acc_trans"] = 1.0
    if pa == 0 and ca == 0 and finb: f["nom_fin"] = 1.0
    if ca == cb and ca != 0: f["same_case"] = 1.0
    if not oka and not okb: f["both_bad"] = 1.0
    if ca == 1 and not possb: f["gen_nonposs"] = 1.0
    return f

def make_cse_morph(tagger, base_morph=None):
    """MAPP-2 morph hook: morph(word, left, right) -> dict. Кандидат сөздің өз талдауы + сол/оң көрші сөзбен CSE жұп-белгілері."""
    def morph(word, left, right):
        out = dict(base_morph(word, left, right)) if base_morph else {}
        t = tagger.word(word); pos, case, poss, fin, mot, tr, ok = t
        out["cse_ok"] = float(ok); out["cse_pos%d" % pos] = 1.0
        if case: out["cse_case%d" % case] = 1.0
        if poss: out["cse_poss"] = 1.0
        if fin: out["cse_fin"] = 1.0
        if left:
            for k, v in _pair_feats(tagger.word(left[-1]), t).items(): out["L_" + k] = v
        if right:
            for k, v in _pair_feats(t, tagger.word(right[0])).items(): out["R_" + k] = v
        return out
    return morph


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- CSE морфология (түбір + жалғау талдаушы) ----------
import re as _re_morph
import numpy as np
from collections import defaultdict as _dd, Counter as _Counter

MOTION = {"бар","кел","кет","шық","кір","қайт","жет","ұш","жүр","жүгір","жүз","көш","аттан","қон","түс","өт","апар","әкел","әкет","орал","жылжы","көтеріл","жақында","ауыс"}
TRANSITIVE = {"көр","ал","бер","жаса","оқы","жаз","айт","же","іш","тап","сал","қой","аш","жап","тыңда","түсін","біл","ұста","сат","кес","жу","таңда","шеш","көтер","тексер","дайында","аяқта","баста","бітір","орында"}
_HOMO = {"i":"і","a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","ј":"і","h":"һ"}
_VPAIR = {"а":"е","е":"а","о":"ө","ө":"о","ұ":"ү","ү":"ұ","ы":"і","і":"ы"}
def _mclean(s): return "".join(_HOMO.get(c, c) for c in str(s).strip().lower())
def _mtwin(e): return "".join(_VPAIR.get(c, c) for c in e)
CASES = ["nom","gen","dat","acc","loc","abl","ins"]
CASE_TAG = {"gen":1,"dat":2,"acc":3,"loc":4,"abl":5,"ınst":6,"ınstr":6,"instr":6,"inst":6}
FINITE = {"pst","per","tpat","ıft","tft","aft","apt","tpt","ppt","mdes","mimp","mcond"}

class Tagger:
    def __init__(self, stems, ana, votes):
        self.S, self.ana, self.votes = stems, ana, votes
        self.maxe = max((len(e) for e in ana), default=0); self._w = {}
    @classmethod
    def build(cls, stems_xlsx, endings_xlsx, uni_counter):
        import pandas as _pd
        st = _pd.read_excel(stems_xlsx, header=None)
        S = {_mclean(v) for v in st.iloc[:, 0].dropna()}
        S = {w for w in S if w and all(c.isalpha() or c == "-" for c in w)}
        en = _pd.read_excel(endings_xlsx, header=None, skiprows=1)
        ana = _dd(set)
        for e, d in zip(en.iloc[:, 0], en.iloc[:, 1]):
            if not isinstance(e, str) or not isinstance(d, str): continue
            tags = frozenset(_re_morph.findall(r"<([^>]+)>", d))
            if not tags: continue
            key = _mclean(e)
            for k in (key, _mtwin(key)):
                if k: ana[k].add(tags)
        ana = {k: list(v) for k, v in ana.items()}
        votes = cls._harvest(S, ana, uni_counter)
        print("CSE: түбір %d, жалғау-үлгі %d" % (len(S), len(ana)))
        return cls(S, ana, votes)
    @staticmethod
    def _harvest(S, ana, uni_counter):
        vb = {e for e, a in ana.items() if all("VB" in t for t in a)}
        nb = {e for e, a in ana.items() if all("NB" in t for t in a)}
        maxe = max((len(e) for e in ana), default=0)
        votes = _dd(lambda: [0, 0])
        for w, n in uni_counter.items():
            if len(w) < 4 or not w.isalpha(): continue
            for k in range(len(w) - 1, 1, -1):
                if len(w) - k > maxe: break
                stem, end = w[:k], w[k:]
                if stem not in S: continue
                if end in vb: votes[stem][0] += n
                elif end in nb: votes[stem][1] += n
        return {k: tuple(v) for k, v in votes.items()}
    def stem_pos(self, s):
        v, n = self.votes.get(s, (0, 0)); t = v + n
        if t < 5: return 2
        r = v / t
        return 1 if r > 0.6 else (0 if r < 0.4 else 2)
    def word(self, w):
        t = self._w.get(w)
        if t is None: t = self._analyze(w); self._w[w] = t
        return t
    def _analyze(self, w):
        if not w or not w.isalpha(): return (2, 0, False, False, False, False, False)
        if w in self.S:
            p = self.stem_pos(w); return (p, 0, False, p == 1, w in MOTION, w in TRANSITIVE, True)
        best, best_sc = None, -1e9
        for k in range(len(w) - 1, 1, -1):
            if len(w) - k > self.maxe: break
            stem, end = w[:k], w[k:]
            if stem not in self.S: continue
            for tags in self.ana.get(end, ()):
                sp = self.stem_pos(stem); sc = 0.1 * len(end)
                if "VB" in tags: sc += 2.0 if sp == 1 else (-1.0 if sp == 0 else 0.0)
                if "NB" in tags: sc += 2.0 if sp == 0 else (-1.0 if sp == 1 else 0.0)
                if sc > best_sc: best, best_sc = (stem, tags, sp), sc
        if best is None: return (2, 0, False, False, False, False, False)
        stem, tags, sp = best
        pos = 1 if ("VB" in tags and "NB" not in tags) else (0 if ("NB" in tags and "VB" not in tags) else sp)
        case = 0
        for t in tags:
            if t in CASE_TAG: case = CASE_TAG[t]; break
        poss = "pos" in tags
        fin = pos == 1 and bool(tags & FINITE) and "pp" not in tags
        return (pos, case, poss, fin, stem in MOTION, stem in TRANSITIVE, True)

CSE_TR_NAMES = (["pos_%s_%s" % (a, b) for a in range(3) for b in range(3)]
                + ["%s_poss" % c for c in CASES] + ["%s_verb" % c for c in CASES]
                + ["dat_motion", "acc_trans", "nom_fin", "same_case", "both_bad", "gen_nonposs"])

def _pair_feats(a, b):
    """(алдыңғы сөз, келесі сөз) талдауынан GEN+POSS, DAT+MOTION, ACC+TRANS т.б. белгілер."""
    pa, ca, _, _, _, _, oka = a; pb, cb, possb, finb, motb, trb, okb = b
    f = {}
    f["pos_%d_%d" % (pa, pb)] = 1.0
    if possb: f["%s_poss" % CASES[ca]] = 1.0
    if pb == 1: f["%s_verb" % CASES[ca]] = 1.0
    if ca == 2 and motb: f["dat_motion"] = 1.0
    if ca == 3 and trb: f["acc_trans"] = 1.0
    if pa == 0 and ca == 0 and finb: f["nom_fin"] = 1.0
    if ca == cb and ca != 0: f["same_case"] = 1.0
    if not oka and not okb: f["both_bad"] = 1.0
    if ca == 1 and not possb: f["gen_nonposs"] = 1.0
    return f

def make_cse_morph(tagger, base_morph=None):
    """MAPP-2 morph hook: morph(word, left, right) -> dict. Кандидат сөздің өз талдауы + сол/оң көрші сөзбен CSE жұп-белгілері."""
    def morph(word, left, right):
        out = dict(base_morph(word, left, right)) if base_morph else {}
        t = tagger.word(word); pos, case, poss, fin, mot, tr, ok = t
        out["cse_ok"] = float(ok); out["cse_pos%d" % pos] = 1.0
        if case: out["cse_case%d" % case] = 1.0
        if poss: out["cse_poss"] = 1.0
        if fin: out["cse_fin"] = 1.0
        if left:
            for k, v in _pair_feats(tagger.word(left[-1]), t).items(): out["L_" + k] = v
        if right:
            for k, v in _pair_feats(t, tagger.word(right[0])).items(): out["R_" + k] = v
        return out
    return morph


# ---------- биграмма ЯМ (корпустан) ----------
import math
from collections import Counter

class BigramLM:
    """Корпустан биграмма ЯМ. MLMScorer-мен бірдей интерфейс: score(left, cand, right) -> log P(cand|left) + log P(right|cand).
    Тек гипотезалардағы сөздерге қатысты биграммалар сақталады (жад үнемі)."""
    def __init__(self, corpus_path, vocab, norm, lam=0.7, max_lines=None, log=print):
        self.uni, self.bi = Counter(), Counter(); V = set(vocab); self.lam = lam
        n = 0
        with open(corpus_path, encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                w = norm(line).split()
                if not w: continue
                self.uni.update(w)
                prev = "<s>"
                for x in w:
                    if prev in V or x in V: self.bi[(prev, x)] += 1
                    prev = x
                if prev in V: self.bi[(prev, "</s>")] += 1
                n += 1
                if max_lines and n >= max_lines: break
        self.ctx = Counter()
        for (a, b), c in self.bi.items(): self.ctx[a] += c
        self.N = sum(self.uni.values()); self.Vn = len(self.uni) + 1
        self.uni["<s>"] = n
        log(f"ЯМ: {n} сөйлем, {self.N} сөз, сөздік {len(self.uni)}, биграмма {len(self.bi)}")
    def p(self, prev, w):
        pu = (self.uni.get(w, 0) + 0.5) / (self.N + 0.5 * self.Vn)
        cp = self.uni.get(prev, 0) if prev != "<s>" else self.uni["<s>"]
        if prev in self.ctx or prev == "<s>":
            tot = self.ctx.get(prev, 0)
            pb = self.bi.get((prev, w), 0) / tot if tot else 0.0
            return math.log(self.lam * pb + (1 - self.lam) * pu)
        return math.log(pu)
    def score(self, left, cand, right):
        l = left[-1] if left else "<s>"
        s = self.p(l, cand)
        s += self.p(cand, right[0]) if right else self.p(cand, "</s>")
        return s


# ---------- қазақша нормализация: цифр→сөз, бөгеу дыбыстар ----------
import re
ONES=["","бір","екі","үш","төрт","бес","алты","жеті","сегіз","тоғыз"]
TENS=["","он","жиырма","отыз","қырық","елу","алпыс","жетпіс","сексен","тоқсан"]
def _lt1000(n):
    w=[]
    if n>=100: w+= ([ONES[n//100]] if n//100>1 else [])+["жүз"]; n%=100
    if n>=10: w.append(TENS[n//10]); n%=10
    if n: w.append(ONES[n])
    return w
def num2kk(n):
    if n==0: return "нөл"
    parts=[]
    for div,name in ((10**9,"миллиард"),(10**6,"миллион"),(1000,"мың")):
        if n>=div:
            q=n//div; n%=div
            parts+= ([] if (name=="мың" and q==1) else _lt1000(q))+[name]
    parts+=_lt1000(n)
    return " ".join(parts)
FILLERS={"ыыы","ы","ыы","мхм","і","ііі","іі","ммм","мм","м","ым","ыым","хм","эмм","ээ"}
def digits_to_words(t):
    def rep(m):
        s=m.group(1)
        return num2kk(int(s)) if len(s)<=12 else s
    return re.sub(r"(\d+)[^\W\d_]*",lambda m: rep(m)+" ",t)   # '90шы' -> 'тоқсан' (қосымша тасталады)
def norm_eval(t, fillers=True, numerals=True):
    t=str(t).lower(); t=re.sub(r"[^\w\s]","",t)
    if numerals: t=digits_to_words(t)
    w=t.split()
    if fillers: w=[x for x in w if x not in FILLERS]
    return " ".join(w)


# ---------- CSE морфология (түбір + жалғау талдаушы) ----------
import re as _re_morph
import numpy as np
from collections import defaultdict as _dd, Counter as _Counter

MOTION = {"бар","кел","кет","шық","кір","қайт","жет","ұш","жүр","жүгір","жүз","көш","аттан","қон","түс","өт","апар","әкел","әкет","орал","жылжы","көтеріл","жақында","ауыс"}
TRANSITIVE = {"көр","ал","бер","жаса","оқы","жаз","айт","же","іш","тап","сал","қой","аш","жап","тыңда","түсін","біл","ұста","сат","кес","жу","таңда","шеш","көтер","тексер","дайында","аяқта","баста","бітір","орында"}
_HOMO = {"i":"і","a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","y":"у","k":"к","m":"м","t":"т","ј":"і","h":"һ"}
_VPAIR = {"а":"е","е":"а","о":"ө","ө":"о","ұ":"ү","ү":"ұ","ы":"і","і":"ы"}
def _mclean(s): return "".join(_HOMO.get(c, c) for c in str(s).strip().lower())
def _mtwin(e): return "".join(_VPAIR.get(c, c) for c in e)
CASES = ["nom","gen","dat","acc","loc","abl","ins"]
CASE_TAG = {"gen":1,"dat":2,"acc":3,"loc":4,"abl":5,"ınst":6,"ınstr":6,"instr":6,"inst":6}
FINITE = {"pst","per","tpat","ıft","tft","aft","apt","tpt","ppt","mdes","mimp","mcond"}

class Tagger:
    def __init__(self, stems, ana, votes):
        self.S, self.ana, self.votes = stems, ana, votes
        self.maxe = max((len(e) for e in ana), default=0); self._w = {}
    @classmethod
    def build(cls, stems_xlsx, endings_xlsx, uni_counter):
        import pandas as _pd
        st = _pd.read_excel(stems_xlsx, header=None)
        S = {_mclean(v) for v in st.iloc[:, 0].dropna()}
        S = {w for w in S if w and all(c.isalpha() or c == "-" for c in w)}
        en = _pd.read_excel(endings_xlsx, header=None, skiprows=1)
        ana = _dd(set)
        for e, d in zip(en.iloc[:, 0], en.iloc[:, 1]):
            if not isinstance(e, str) or not isinstance(d, str): continue
            tags = frozenset(_re_morph.findall(r"<([^>]+)>", d))
            if not tags: continue
            key = _mclean(e)
            for k in (key, _mtwin(key)):
                if k: ana[k].add(tags)
        ana = {k: list(v) for k, v in ana.items()}
        votes = cls._harvest(S, ana, uni_counter)
        print("CSE: түбір %d, жалғау-үлгі %d" % (len(S), len(ana)))
        return cls(S, ana, votes)
    @staticmethod
    def _harvest(S, ana, uni_counter):
        vb = {e for e, a in ana.items() if all("VB" in t for t in a)}
        nb = {e for e, a in ana.items() if all("NB" in t for t in a)}
        maxe = max((len(e) for e in ana), default=0)
        votes = _dd(lambda: [0, 0])
        for w, n in uni_counter.items():
            if len(w) < 4 or not w.isalpha(): continue
            for k in range(len(w) - 1, 1, -1):
                if len(w) - k > maxe: break
                stem, end = w[:k], w[k:]
                if stem not in S: continue
                if end in vb: votes[stem][0] += n
                elif end in nb: votes[stem][1] += n
        return {k: tuple(v) for k, v in votes.items()}
    def stem_pos(self, s):
        v, n = self.votes.get(s, (0, 0)); t = v + n
        if t < 5: return 2
        r = v / t
        return 1 if r > 0.6 else (0 if r < 0.4 else 2)
    def word(self, w):
        t = self._w.get(w)
        if t is None: t = self._analyze(w); self._w[w] = t
        return t
    def _analyze(self, w):
        if not w or not w.isalpha(): return (2, 0, False, False, False, False, False)
        if w in self.S:
            p = self.stem_pos(w); return (p, 0, False, p == 1, w in MOTION, w in TRANSITIVE, True)
        best, best_sc = None, -1e9
        for k in range(len(w) - 1, 1, -1):
            if len(w) - k > self.maxe: break
            stem, end = w[:k], w[k:]
            if stem not in self.S: continue
            for tags in self.ana.get(end, ()):
                sp = self.stem_pos(stem); sc = 0.1 * len(end)
                if "VB" in tags: sc += 2.0 if sp == 1 else (-1.0 if sp == 0 else 0.0)
                if "NB" in tags: sc += 2.0 if sp == 0 else (-1.0 if sp == 1 else 0.0)
                if sc > best_sc: best, best_sc = (stem, tags, sp), sc
        if best is None: return (2, 0, False, False, False, False, False)
        stem, tags, sp = best
        pos = 1 if ("VB" in tags and "NB" not in tags) else (0 if ("NB" in tags and "VB" not in tags) else sp)
        case = 0
        for t in tags:
            if t in CASE_TAG: case = CASE_TAG[t]; break
        poss = "pos" in tags
        fin = pos == 1 and bool(tags & FINITE) and "pp" not in tags
        return (pos, case, poss, fin, stem in MOTION, stem in TRANSITIVE, True)

CSE_TR_NAMES = (["pos_%s_%s" % (a, b) for a in range(3) for b in range(3)]
                + ["%s_poss" % c for c in CASES] + ["%s_verb" % c for c in CASES]
                + ["dat_motion", "acc_trans", "nom_fin", "same_case", "both_bad", "gen_nonposs"])

def _pair_feats(a, b):
    """(алдыңғы сөз, келесі сөз) талдауынан GEN+POSS, DAT+MOTION, ACC+TRANS т.б. белгілер."""
    pa, ca, _, _, _, _, oka = a; pb, cb, possb, finb, motb, trb, okb = b
    f = {}
    f["pos_%d_%d" % (pa, pb)] = 1.0
    if possb: f["%s_poss" % CASES[ca]] = 1.0
    if pb == 1: f["%s_verb" % CASES[ca]] = 1.0
    if ca == 2 and motb: f["dat_motion"] = 1.0
    if ca == 3 and trb: f["acc_trans"] = 1.0
    if pa == 0 and ca == 0 and finb: f["nom_fin"] = 1.0
    if ca == cb and ca != 0: f["same_case"] = 1.0
    if not oka and not okb: f["both_bad"] = 1.0
    if ca == 1 and not possb: f["gen_nonposs"] = 1.0
    return f

def make_cse_morph(tagger, base_morph=None):
    """MAPP-2 morph hook: morph(word, left, right) -> dict. Кандидат сөздің өз талдауы + сол/оң көрші сөзбен CSE жұп-белгілері."""
    def morph(word, left, right):
        out = dict(base_morph(word, left, right)) if base_morph else {}
        t = tagger.word(word); pos, case, poss, fin, mot, tr, ok = t
        out["cse_ok"] = float(ok); out["cse_pos%d" % pos] = 1.0
        if case: out["cse_case%d" % case] = 1.0
        if poss: out["cse_poss"] = 1.0
        if fin: out["cse_fin"] = 1.0
        if left:
            for k, v in _pair_feats(tagger.word(left[-1]), t).items(): out["L_" + k] = v
        if right:
            for k, v in _pair_feats(t, tagger.word(right[0])).items(): out["R_" + k] = v
        return out
    return morph


# ---------- биграмма ЯМ (корпустан) ----------
import math
from collections import Counter

class BigramLM:
    """Корпустан биграмма ЯМ. MLMScorer-мен бірдей интерфейс: score(left, cand, right) -> log P(cand|left) + log P(right|cand).
    Тек гипотезалардағы сөздерге қатысты биграммалар сақталады (жад үнемі)."""
    def __init__(self, corpus_path, vocab, norm, lam=0.7, max_lines=None, log=print):
        self.uni, self.bi = Counter(), Counter(); V = set(vocab); self.lam = lam
        n = 0
        with open(corpus_path, encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                w = norm(line).split()
                if not w: continue
                self.uni.update(w)
                prev = "<s>"
                for x in w:
                    if prev in V or x in V: self.bi[(prev, x)] += 1
                    prev = x
                if prev in V: self.bi[(prev, "</s>")] += 1
                n += 1
                if max_lines and n >= max_lines: break
        self.ctx = Counter()
        for (a, b), c in self.bi.items(): self.ctx[a] += c
        self.N = sum(self.uni.values()); self.Vn = len(self.uni) + 1
        self.uni["<s>"] = n
        log(f"ЯМ: {n} сөйлем, {self.N} сөз, сөздік {len(self.uni)}, биграмма {len(self.bi)}")
    def p(self, prev, w):
        pu = (self.uni.get(w, 0) + 0.5) / (self.N + 0.5 * self.Vn)
        cp = self.uni.get(prev, 0) if prev != "<s>" else self.uni["<s>"]
        if prev in self.ctx or prev == "<s>":
            tot = self.ctx.get(prev, 0)
            pb = self.bi.get((prev, w), 0) / tot if tot else 0.0
            return math.log(self.lam * pb + (1 - self.lam) * pu)
        return math.log(pu)
    def score(self, left, cand, right):
        l = left[-1] if left else "<s>"
        s = self.p(l, cand)
        s += self.p(cand, right[0]) if right else self.p(cand, "</s>")
        return s


Writing mapp2_core.py


In [3]:
import os, glob, pandas as pd, numpy as np, time, sys
sys.path.insert(0, "/content")
from mapp2_core import *
_cands = ["/content/drive/MyDrive/Colab Notebooks/dataset_wav16k", "/content/drive/MyDrive/dataset_wav16k"] + glob.glob("/content/drive/MyDrive/**/dataset_wav16k", recursive=True)
BASE = next((c for c in _cands if os.path.isdir(c) and (glob.glob(c + "/*.csv") or os.path.isdir(c + "/ksc2"))), _cands[0])
print("BASE:", BASE)
OUT_DIR = os.path.join(BASE, "mapp2"); os.makedirs(OUT_DIR, exist_ok=True)

# (атау, файл, салмақ). ПЕРВЫЙ = негізгі жүйе (pivot, ең күшті RAW).
SYSTEMS = [
    ("ksc2_vad",      "ksc2/mapp_v2_batch_results_ksc2_vad.csv",    1.0),
    ("kkturbo_vad",   "mapp_v2_batch_results_kkturbo_vad.csv",      0.9),
    ("kkturbo_chunk", "mapp_v2_batch_results_kkturbo_chunk.csv",    0.8),
    ("ksc2_chunk",    "ksc2/mapp_v2_batch_results_ksc2_chunk.csv",  0.7),
    ("mms_vad",       "mms/mapp_v2_batch_results_mms_vad.csv",      0.7),
    ("ctc2_vad",      "ctc2/mapp_v2_batch_results_xlsr_kk_vad.csv", 0.6),   # қосымша CTC жүйесі (жоқ болса өткізіледі)
]
USE_MLM   = False                # MLM әзірге өшірулі (Stage 6 жасалмайды); True = MLM белгісін қосу
MLM_MODEL = "xlm-roberta-base"
FOLDS = 5

# CSV жоқ болса, xlsx-тің per_file парағынан оқиды (glob арқылы Drive ішінен іздейді)
ALT = {
    "ksc2_vad":      ("ksc2/mapp_asr_ksc2_comparison.xlsx", "ksc2_vad_per_file"),
    "ksc2_chunk":    ("ksc2/mapp_asr_ksc2_comparison.xlsx", "ksc2_chunk_per_file"),
    "mms_vad":       ("mms/mapp_asr_mms_comparison.xlsx",   "mms_per_file"),
    "kkturbo_chunk": ("mapp_asr_kkturbo_vad_comparison.xlsx", "kkturbo_chunk_per_file"),
}
def load(name, path):
    p = os.path.join(BASE, path)
    if os.path.exists(p):
        d = pd.read_csv(p)
    elif name in ALT:
        hits = glob.glob(os.path.join(BASE, "**", os.path.basename(ALT[name][0])), recursive=True)
        if not hits: print("  жоқ, өткізілді:", path); return None
        d = pd.read_excel(hits[0], sheet_name=ALT[name][1]); print("  xlsx-тен оқылды:", hits[0])
    else:
        print("  жоқ, өткізілді:", path); return None
    return {(str(r.duration), str(r.stem)): r for r in d.itertuples()}

SRC = {}
USE_CTC2 = False   # xlsr-kk пайда бермеді (0.1795 vs 0.1783) → негізгі нұсқада өшірулі
for name, path, w in SYSTEMS:
    if name == "ctc2_vad" and not USE_CTC2: continue
    s = load(name, path)
    if s is not None: SRC[name] = (s, w); print("OK", name, len(s))
names = list(SRC)
assert len(names) >= 2, "кемінде 2 жүйе керек"
keys = [k for k in SRC[names[0]][0] if all(k in SRC[n][0] for n in names)]
files = [f"{a}/{b}" for a, b in keys]
pivot = SRC[names[0]][0]
NORM_FILLERS = True    # «ыыы, мхм, ііі…» бөгеу дыбыстарды ref және гипотезадан алып тастау
NORM_NUM     = True    # цифр → қазақша сөз (екі жақта бірдей)
EXCLUDE      = []   # ref түзетілді, шығарылатын файл жоқ. (қажет болса: ["10m/10m_009"])
def get_ref(a, b):
    """Жаңартылған анықтама: references/<ұзындық>/<файл>.txt (CSV ішіндегі ref ескірген болуы мүмкін)."""
    p = os.path.join(BASE, "references", a, b + ".txt")
    if os.path.exists(p): return open(p, encoding="utf-8").read().strip(), True
    return str(pivot[(a, b)].ref), False
_r = {k: get_ref(*k) for k in keys}
refs = {f"{a}/{b}": norm_eval(_r[(a, b)][0], NORM_FILLERS, NORM_NUM) for a, b in keys}
_chg = [f"{a}/{b}" for a, b in keys if norm_eval(_r[(a, b)][0], NORM_FILLERS, NORM_NUM) != norm_eval(pivot[(a, b)].ref, NORM_FILLERS, NORM_NUM)]
print("references/*.txt-тен оқылды:", sum(v[1] for v in _r.values()), "/", len(keys), "| CSV-дегі ref-тен өзгерген:", _chg)
print("ортақ файл:", len(files), "| жүйелер:", names)

BASE: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k
OK ksc2_vad 100
OK kkturbo_vad 100
OK kkturbo_chunk 100
OK ksc2_chunk 100
OK mms_vad 100
references/*.txt-тен оқылды: 100 / 100 | CSV-дегі ref-тен өзгерген: ['10m/10m_009']
ортақ файл: 100 | жүйелер: ['ksc2_vad', 'kkturbo_vad', 'kkturbo_chunk', 'ksc2_chunk', 'mms_vad']


In [4]:
import numpy as np, pandas as pd, pickle
import mapp2_core as m
names_use = names
N_ = np.array([len(refs[f].split()) for f in files]); NN = N_.sum()
def E(h): return np.array([m.errs(refs[f], h[f]) for f in files])
hyps_all = {f"{a}/{b}": [dedup_full(norm_eval(SRC[n][0][(a, b)].raw, NORM_FILLERS, NORM_NUM)) for n in names_use] for a, b in keys}
SCHEMES = {"paper": [1.0, 0.9, 0.8, 0.7, 0.7], "equal": [1, 1, 1, 1, 1], "S3": [1, 1, .5, .5, .5], "S4": [1, .8, .6, .4, .4], "S5": [1, 1, 1, .5, .5]}
assert len(names_use) == 5, names_use
outs = {}
for k, sw_ in SCHEMES.items():
    out_, tu_, _ = m.run_cv(files, refs, hyps_all, sw_, log=lambda *a: None)
    outs[k] = out_
    print(f"{k:6s} {sw_}  vote {E(out_['vote']).sum()/NN:.4f}  CRF {E(out_['crf_auto']).sum()/NN:.4f}  tau {tu_}")
# вложенная таңдау: әр сыртқы фолдта схема тек оқыту файлдарындағы дауыс беру қатесі бойынша таңдалады
folds = m.split_folds(files, 5)
ev = {k: {f: m.errs(refs[f], outs[k]["vote"][f]) for f in files} for k in SCHEMES}
sel = {"vote": {}, "crf_auto": {}}; chosen = []
for te in folds:
    tr = [f for f in files if f not in set(te)]
    best = min(SCHEMES, key=lambda k: sum(ev[k][f] for f in tr)); chosen.append(best)
    for f in te:
        sel["vote"][f] = outs[best]["vote"][f]; sel["crf_auto"][f] = outs[best]["crf_auto"][f]
print("фолд бойынша таңдалған схема:", chosen)
print(f"CV-таңдалған: vote {E(sel['vote']).sum()/NN:.4f}  CRF {E(sel['crf_auto']).sum()/NN:.4f}")
rng = np.random.default_rng(0); idx = rng.integers(0, len(files), (10000, len(files)))
def bs(a, b):
    d = (a[idx].sum(1) - b[idx].sum(1)) / N_[idx].sum(1) * 100
    return round(float(d.mean()), 3), np.round(np.percentile(d, [2.5, 97.5]), 3)
print("CRF: тең салмақ − мақала салмағы (п.п.):", bs(E(outs['equal']['crf_auto']), E(outs['paper']['crf_auto'])))
print("тең салмақ: дауыс беру − CRF (CRF ұтысы):", bs(E(outs['equal']['vote']), E(outs['equal']['crf_auto'])))
print("S5: дауыс беру − CRF:", bs(E(outs['S5']['vote']), E(outs['S5']['crf_auto'])))

paper  [1.0, 0.9, 0.8, 0.7, 0.7]  vote 0.1828  CRF 0.1793  tau [0.5, 0.5, 0.5, 0.5, 0.5]
equal  [1, 1, 1, 1, 1]  vote 0.1821  CRF 0.1803  tau [0.5, 0.5, 0.4, 0.5, 0.5]
S3     [1, 1, 0.5, 0.5, 0.5]  vote 0.1845  CRF 0.1816  tau [0.4, 0.4, 0.4, 0.4, 0.4]
S4     [1, 0.8, 0.6, 0.4, 0.4]  vote 0.1843  CRF 0.1844  tau [0.5, 0.5, 0.4, 0.4, 0.6]
S5     [1, 1, 1, 0.5, 0.5]  vote 0.1799  CRF 0.1795  tau [0.5, 0.5, 0.5, 0.5, 0.5]
фолд бойынша таңдалған схема: ['S5', 'S5', 'S5', 'S5', 'S5']
CV-таңдалған: vote 0.1799  CRF 0.1795
CRF: тең салмақ − мақала салмағы (п.п.): (0.098, array([-0.015,  0.202]))
тең салмақ: дауыс беру − CRF (CRF ұтысы): (0.177, array([0.104, 0.257]))
S5: дауыс беру − CRF: (0.04, array([0.003, 0.078]))


In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
LAB = ["C0", "C1", "C2", "C3"]
class Flat:
    """CRF-тің орнына әр слотты дербес жіктейтін модель (контекстсіз). Белгілер бірдей."""
    def __init__(self, kind): self.kind = kind
    def fit(self, X, Y):
        Xf = [x for s in X for x in s]; Yf = [y for s in Y for y in s]
        self.dv = DictVectorizer(sparse=(self.kind == "lr")); A = self.dv.fit_transform(Xf)
        self.c = LogisticRegression(C=1.0, max_iter=300) if self.kind == "lr" else HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=4, random_state=0)
        self.c.fit(A, Yf); return self
    def predict_marginals_single(self, xs):
        P = self.c.predict_proba(self.dv.transform(xs)); cl = list(self.c.classes_)
        return [{l: (float(p[cl.index(l)]) if l in cl else 0.0) for l in LAB} for p in P]
SW = [1.0, 0.9, 0.8, 0.7, 0.7]
orig_train = m.train_crf
res2 = {}
for kind in ("crf", "lr", "gb"):
    m.train_crf = orig_train if kind == "crf" else (lambda kk: (lambda data, fl: Flat(kk).fit([data[f][1] for f in fl], [data[f][2] for f in fl])))(kind)
    out_, tu_, _ = m.run_cv(files, refs, hyps_all, SW, log=lambda *a: None)
    res2[kind] = out_
    print(f"{kind:4s} vote {E(out_['vote']).sum()/NN:.4f}  селектор(guard) {E(out_['crf_auto']).sum()/NN:.4f}  tau {tu_}")
m.train_crf = orig_train
ec = E(res2["crf"]["crf_auto"]); evv = E(res2["crf"]["vote"])
print("CRF   − дауыс беру:", bs(evv, ec))
for k in ("lr", "gb"):
    e = E(res2[k]["crf_auto"]); print(k, "ұтыс дауыс берумен салыстырғанда:", bs(evv, e), "| CRF артықшылығы:", bs(e, ec))

crf  vote 0.1828  селектор(guard) 0.1793  tau [0.5, 0.5, 0.5, 0.5, 0.5]
lr   vote 0.1828  селектор(guard) 0.1800  tau [0.5, 0.4, 0.4, 0.4, 0.4]
gb   vote 0.1828  селектор(guard) 0.1797  tau [0.5, 0.6, 0.5, 0.5, 0.5]
CRF   − дауыс беру: (0.346, array([0.238, 0.46 ]))
lr ұтыс дауыс берумен салыстырғанда: (0.276, array([0.158, 0.399])) | CRF артықшылығы: (0.07, array([-0.012,  0.156]))
gb ұтыс дауыс берумен салыстырғанда: (0.306, array([0.19 , 0.428])) | CRF артықшылығы: (0.04, array([-0.043,  0.121]))


In [6]:
# 3-тәжірибе: жүйе салмағы әр фолдта ТЕК оқыту файлдарындағы жеке жүйелердің WER-інен есептеледі (w = min(WER)/WER_i)
taus = (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
folds3 = m.split_folds(files, 5)
out3 = {"vote": {}, "crf_auto": {}}; used = []
for te in folds3:
    tr = [f for f in files if f not in set(te)]
    wer = [sum(m.errs(refs[f], hyps_all[f][i]) for f in tr) / sum(len(refs[f].split()) for f in tr) for i in range(5)]
    sw_ = [min(wer) / w for w in wer]; used.append([round(x, 2) for x in sw_])
    data = {}
    for f in files:
        net = m.build_network(hyps_all[f]); cps, X = m.featurize(net, sw_, None, m.default_morph, None)
        data[f] = (cps, X, m.align_ref(cps, refs[f].split()))
    crf = m.train_crf(data, tr); tau = m.choose_tau(data, refs, tr, taus)
    for f in te:
        cps, X, _ = data[f]; mg = crf.predict_marginals_single(X)
        out3["vote"][f] = m.vote_text(cps); out3["crf_auto"][f] = m.decode(cps, mg, tau)
print("фолд салмақтары:", used)
e_v = E(out3["vote"]); e_c = E(out3["crf_auto"])
print(f"vote {e_v.sum()/NN:.4f}  CRF {e_c.sum()/NN:.4f}  ({e_v.sum()} / {e_c.sum()} қате)")
print("дауыс беру − CRF (п.п., 95% СИ):", bs(e_v, e_c))

фолд салмақтары: [[1.0, 0.98, 0.7, 0.43, 0.4], [1.0, 0.95, 0.7, 0.44, 0.4], [1.0, 0.95, 0.71, 0.44, 0.41], [1.0, 0.95, 0.73, 0.45, 0.41], [1.0, 0.95, 0.7, 0.44, 0.41]]
vote 0.1801  CRF 0.1793  (5467 / 5445 қате)
дауыс беру − CRF (п.п., 95% СИ): (0.073, array([0.034, 0.115]))


In [7]:
rows = [{"experiment": "weights", "scheme": k, "vote_WER": E(outs[k]["vote"]).sum()/NN, "CRF_WER": E(outs[k]["crf_auto"]).sum()/NN} for k in SCHEMES]
rows.append({"experiment": "weights", "scheme": "CV-chosen", "vote_WER": E(sel["vote"]).sum()/NN, "CRF_WER": E(sel["crf_auto"]).sum()/NN})
rows += [{"experiment": "selector", "scheme": k, "vote_WER": E(res2[k]["vote"]).sum()/NN, "CRF_WER": E(res2[k]["crf_auto"]).sum()/NN} for k in res2]
df = pd.DataFrame(rows).round(4); print(df.to_string(index=False))
df.to_excel(os.path.join(OUT_DIR, "ablation_weights_selector.xlsx"), index=False); print("Сақталды:", OUT_DIR)

experiment    scheme  vote_WER  CRF_WER
   weights     paper    0.1828   0.1793
   weights     equal    0.1821   0.1803
   weights        S3    0.1845   0.1816
   weights        S4    0.1843   0.1844
   weights        S5    0.1799   0.1795
   weights CV-chosen    0.1799   0.1795
  selector       crf    0.1828   0.1793
  selector        lr    0.1828   0.1800
  selector        gb    0.1828   0.1797
Сақталды: /content/drive/MyDrive/Colab Notebooks/dataset_wav16k/mapp2


In [9]:
# 4-тексеру: дәл қате сандары (дөңгелектеуді тексеру). Алдымен 3–6 ұяшықтар орындалған болуы керек.
# (а) 1/WER салмақтары — 6-ұяшық шығысындағы "vote .. / CRF .. қате" жолы; мұнда қайта қорытындылаймыз:
print("1/WER: дауыс беру", int(e_v.sum()), "қате, CRF", int(e_c.sum()), "қате, N =", NN,
      "| айырма п.п. =", round((e_v.sum() - e_c.sum()) / NN * 100, 4), "| WER:", round(e_v.sum()/NN*100, 4), round(e_c.sum()/NN*100, 4))

# (б) белгі шуы: эталон сөзі кандидаттар арасында жоқ слоттарды оқытудан алып тастау
def align_flag(cps, ref):
    """m.align_ref-пен бірдей, бірақ эталон сөзі жоқ слотқа None қайтарады."""
    n, mm = len(cps), len(ref); INF = 10**9
    words = [[c["w"] for c in cs] for cs in cps]
    D = [[INF]*(mm+1) for _ in range(n+1)]; B = [[None]*(mm+1) for _ in range(n+1)]; D[0][0] = 0
    for i in range(n+1):
        for j in range(mm+1):
            d = D[i][j]
            if d >= INF: continue
            if i < n and j < mm:
                c = d + (0 if ref[j] in words[i] else 1)
                if c < D[i+1][j+1]: D[i+1][j+1], B[i+1][j+1] = c, "M"
            if i < n:
                c = d + (0 if m.EPS in words[i] else 1)
                if c < D[i+1][j]: D[i+1][j], B[i+1][j] = c, "D"
            if j < mm:
                c = d + 1
                if c < D[i][j+1]: D[i][j+1], B[i][j+1] = c, "I"
    gold, i, j = [None]*n, n, mm
    while i > 0 or j > 0:
        op = B[i][j]
        if op == "M": gold[i-1] = ref[j-1]; i -= 1; j -= 1
        elif op == "D": gold[i-1] = m.EPS; i -= 1
        else: j -= 1
    return [f"C{ws.index(g)}" if g in ws else None for ws, g in zip(words, gold)]

SW = [1.0, 0.9, 0.8, 0.7, 0.7]
FLAG = {}
for f in files:
    net = m.build_network(hyps_all[f]); cps, _ = m.featurize(net, SW, None, m.default_morph, None)
    FLAG[f] = align_flag(cps, refs[f].split())
tot = sum(len(v) for v in FLAG.values()); absent = sum(x is None for v in FLAG.values() for x in v)
print("слот:", tot, "| эталон сөзі жоқ:", absent, f"({absent/tot*100:.1f}%)")

import sklearn_crfsuite
def train_clean(data, fl):
    """Тізбектер эталон сөзі жоқ слоттарда ҚИЫЛАДЫ (слоттар қолданылмайды, көрші бөліктер бір-біріне жалғанбайды)."""
    X, Y = [], []
    for f in fl:
        _, Xf, Yf = data[f]
        segX, segY = [], []
        for i, l in enumerate(FLAG[f]):
            if l is None:
                if segX: X.append(segX); Y.append(segY); segX, segY = [], []
            else:
                segX.append(Xf[i]); segY.append(Yf[i])
        if segX: X.append(segX); Y.append(segY)
    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=200, all_possible_transitions=True)
    crf.fit(X, Y); return crf

orig_train = m.train_crf
out_base, _, _ = m.run_cv(files, refs, hyps_all, SW, log=lambda *a: None)
m.train_crf = train_clean
out_clean, _, _ = m.run_cv(files, refs, hyps_all, SW, log=lambda *a: None)
m.train_crf = orig_train
eb, ecl, evt = E(out_base["crf_auto"]), E(out_clean["crf_auto"]), E(out_base["vote"])
print("дауыс беру:", int(evt.sum()), "қате |", round(evt.sum()/NN*100, 4))
print("CRF (базалық):", int(eb.sum()), "қате |", round(eb.sum()/NN*100, 4))
print("CRF (шусыз оқыту):", int(ecl.sum()), "қате |", round(ecl.sum()/NN*100, 4))
print("айырма (базалық − шусыз), п.п.:", round((eb.sum() - ecl.sum())/NN*100, 4), "| бутстреп:", bs(eb, ecl))


1/WER: дауыс беру 5467 қате, CRF 5445 қате, N = 30363 | айырма п.п. = 0.0725 | WER: 18.0055 17.933
слот: 32614 | эталон сөзі жоқ: 2798 (8.6%)
дауыс беру: 5550 қате | 18.2788
CRF (базалық): 5445 қате | 17.933
CRF (шусыз оқыту): 5431 қате | 17.8869
айырма (базалық − шусыз), п.п.: 0.0461 | бутстреп: (0.047, array([-0.031,  0.136]))
